In [6]:
# RAG-Powered-Document-QA-System
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os

In [10]:
load_dotenv()
openai_key = os.getenv("ApiKey")

if not openai_key:
    print("API key not loaded!")
else:
    print("API key loaded successfully.")
    

# Vector Embeddings 
embeddings = OpenAIEmbeddings(model="text-embedding-3-small",
                                openai_api_key=openai_key)

# Create local database
persist_directory = "./chroma_db"

if os.path.exists(persist_directory):
    # ✅ LOAD EXISTING EMBEDDINGS
    print("📂 Loading existing embeddings from disk...")
    vector_db = Chroma(persist_directory=persist_directory,
                       embedding_function=embeddings)
    print("✅ Loaded from ./chroma_db/")
    
else:
    # ✅ CREATE NEW EMBEDDINGS
    print("📝 Creating new embeddings...")
    
    # Load PDF
    loader = PyPDFLoader("Pandas Notes.pdf")
    doc = loader.load()
    print(f"✅ Loaded {len(doc)} pages")
    
    # Split into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500, 
        chunk_overlap=50
    )
    chunks = text_splitter.split_documents(doc)
    print(f"✅ Created {len(chunks)} chunks")
    
    # Create and save vector DB
    vector_db = Chroma.from_documents(chunks, 
                                      embeddings,
                                      persist_directory=persist_directory)

    print("✅ Saved to ./chroma_db/")

# Define the retriever
retriever = vector_db.as_retriever(search_kwargs={"k":3})

# 4. Set up LLM and Prompt
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0,
                        openai_api_key=openai_key)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using context. Say 'I don't know' if unsure.\n\n{context}"),
    ("human", "{input}"),
])

# 5. Build and run the RAG chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

response = rag_chain.invoke({"input": "Get all the DataFrame Methods from the given Pdf."})
print(response["answer"])

API key loaded successfully.
📂 Loading existing embeddings from disk...
✅ Loaded from ./chroma_db/
The DataFrame Methods mentioned in the provided context are:

1. df.head(n) - Shows the first n rows (default = 5)
2. df.tail(n) - Shows the last n rows (default = 5)
3. df.sample(n) - Shows random n rows (default = 1)
4. df.info() - Displays column names, data types, memory usage
5. df.describe(n) - (not specified in the context)
6. df.nunique(n) - (not specified in the context)
7. df.shape - (not specified in the context)
8. df.columns - (not specified in the context)
9. df.dtypes - (not specified in the context)
